<a href="https://colab.research.google.com/github/AyazN/DLS-final-project/blob/fix/integrate-search-pipeline/notebooks/full_search_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Model Search — Full Search Pipeline

This demonstration notebook connects the existing project artifacts into one end-to-end pipeline:

**processed metadata → precomputed embeddings → FAISS → BM25 + dense retrieval → RRF → cross-encoder → evaluation**

The notebook does not regenerate embeddings. It uses a low-memory mode by default and automatically caches the completed BM25 retriever on Google Drive before continuing. Colab High-RAM remains preferable for the full-quality configuration.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 1. Configuration

If the integration branch has already been merged, change `REPO_BRANCH` to `develop` or `main`.
The Drive configuration supports either an extracted encoder directory or the `.tar` archive produced by the earlier notebook.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/AyazN/DLS-final-project.git"
REPO_BRANCH = "fix/integrate-search-pipeline"
PROJECT_ROOT = Path("/content/DLS-final-project")

DRIVE_ROOT = Path("/content/drive/MyDrive/Inno/DLS/AISE")

MODEL_DIR_NAME = "sentence-transformers__all-MiniLM-L6-v2"
# To use BGE, switch to:
# MODEL_DIR_NAME = "BAAI__bge-small-en-v1.5"

DRIVE_EMBEDDING_CANDIDATES = [
    DRIVE_ROOT / "embeddings" / MODEL_DIR_NAME,
    DRIVE_ROOT / "data" / "processed" / "embeddings" / MODEL_DIR_NAME,
]
DRIVE_EMBEDDING_ARCHIVE_CANDIDATES = [
    DRIVE_ROOT / "embeddings" / "minilm_embeddings_600k.tar",
    DRIVE_ROOT / "minilm_embeddings_600k.tar",
]

LOCAL_EMBEDDING_DIR = (
    PROJECT_ROOT / "data" / "processed" / "embeddings" / MODEL_DIR_NAME
)
LOCAL_CLEAN_DATASET = PROJECT_ROOT / "data" / "processed" / "clean_dataset.parquet"

CLEAN_DATASET_CANDIDATES = [
    DRIVE_ROOT / "data" / "processed" / "clean_dataset.parquet",
    DRIVE_ROOT / "data" / "clean_dataset.parquet",
    DRIVE_ROOT / "clean_dataset.parquet",
]

INDEX_FILENAME = (
    "minilm_flat_ip.faiss"
    if MODEL_DIR_NAME.startswith("sentence-transformers")
    else "bge_small_flat_ip.faiss"
)
LOCAL_INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / INDEX_FILENAME
DRIVE_INDEX_PATH = DRIVE_ROOT / "indices" / INDEX_FILENAME
DRIVE_INDEX_CANDIDATES = [
    DRIVE_INDEX_PATH,
    DRIVE_ROOT / "data" / "indexes" / INDEX_FILENAME,
    DRIVE_ROOT / "data" / "processed" / "indexes" / INDEX_FILENAME,
]

RELEVANCE_PATH = DRIVE_ROOT / "evaluation" / "relevance.csv"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"

RETRIEVAL_TOP_K = 100
RERANK_TOP_K = 20

# Safe defaults for a free Colab runtime with about 12.7 GB system RAM.
LOW_MEMORY_MODE = True
BODY_MAX_CHARS_OVERRIDE = 200
BM25_CACHE_COMPRESSION = 3


## 2. Repository and dependencies


In [ ]:
import os
import subprocess
import sys

if (PROJECT_ROOT / ".git").exists():
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "fetch", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "checkout", REPO_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            REPO_URL,
            str(PROJECT_ROOT),
        ],
        check=True,
    )

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Repository:", PROJECT_ROOT)
print("Branch:", REPO_BRANCH)


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import platform

import numpy as np
import pandas as pd
import torch

from aise.contracts import EvaluationExample, Query
from aise.evaluation import RetrievalEvaluator
from aise.pipeline import SearchPipeline
from aise.retrieval import (
    BM25Retriever,
    CrossEncoderReranker,
    DenseRetriever,
    HybridRetriever,
    ReciprocalRankFusion,
)
from aise.search_index import (
    FaissFlatIndex,
    load_embedding_artifacts,
    load_search_documents,
)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Torch device:", "cuda" if torch.cuda.is_available() else "cpu")


## 3. Prepare artifacts from Google Drive

The embedding directory must contain `embeddings.npy`, `ids.npy`, `metadata.parquet`, `run_config.json`, `progress.json`, and `text_representation_samples.json`.


In [ ]:
import shutil

REQUIRED_ARTIFACTS = (
    "embeddings.npy",
    "ids.npy",
    "metadata.parquet",
    "run_config.json",
    "progress.json",
    "text_representation_samples.json",
)


def has_artifacts(directory: Path) -> bool:
    return all((directory / name).exists() for name in REQUIRED_ARTIFACTS)


if not has_artifacts(LOCAL_EMBEDDING_DIR):
    LOCAL_EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)
    source_embedding_dir = next(
        (path for path in DRIVE_EMBEDDING_CANDIDATES if has_artifacts(path)),
        None,
    )
    source_archive = next(
        (path for path in DRIVE_EMBEDDING_ARCHIVE_CANDIDATES if path.exists()),
        None,
    )

    if source_embedding_dir is not None:
        print("Copying embedding artifacts from:", source_embedding_dir)
        for name in REQUIRED_ARTIFACTS:
            shutil.copy2(
                source_embedding_dir / name,
                LOCAL_EMBEDDING_DIR / name,
            )
    elif source_archive is not None:
        print("Extracting embedding archive from:", source_archive)
        subprocess.run(
            [
                "tar",
                "-xf",
                str(source_archive),
                "-C",
                str(PROJECT_ROOT),
            ],
            check=True,
        )
    else:
        raise FileNotFoundError(
            "Embedding directory/archive was not found on Google Drive. "
            "Update DRIVE_EMBEDDING_CANDIDATES."
        )

if not LOCAL_CLEAN_DATASET.exists():
    source_dataset = next(
        (path for path in CLEAN_DATASET_CANDIDATES if path.exists()),
        None,
    )
    if source_dataset is None:
        raise FileNotFoundError(
            "clean_dataset.parquet was not found. Update CLEAN_DATASET_CANDIDATES."
        )
    LOCAL_CLEAN_DATASET.parent.mkdir(parents=True, exist_ok=True)
    print("Copying processed metadata from:", source_dataset)
    shutil.copy2(source_dataset, LOCAL_CLEAN_DATASET)

assert has_artifacts(LOCAL_EMBEDDING_DIR)
assert LOCAL_CLEAN_DATASET.exists()

print("Embedding directory:", LOCAL_EMBEDDING_DIR)
print("Processed metadata:", LOCAL_CLEAN_DATASET)


## 4. Load and validate embedding artifacts


In [ ]:
artifacts = load_embedding_artifacts(
    LOCAL_EMBEDDING_DIR,
    mmap_embeddings=True,
    expected_dim=384,
)

print("Shape:", artifacts.embeddings.shape)
print("Dtype:", artifacts.embeddings.dtype)
print("Encoder:", artifacts.model_name)
print("Normalized:", artifacts.normalized)
print("Metadata columns:", artifacts.metadata.columns.tolist())

ENCODER_MODEL_NAME = artifacts.model_name
NORMALIZE_QUERY_EMBEDDINGS = artifacts.normalized
ARTIFACT_ROW_COUNT = artifacts.embeddings.shape[0]
ARTIFACT_DIM = artifacts.embeddings.shape[1]
BODY_MAX_CHARS = (
    BODY_MAX_CHARS_OVERRIDE
    if BODY_MAX_CHARS_OVERRIDE is not None
    else int(artifacts.run_config.get("max_model_card_chars", 2500))
)

print("Low-memory mode:", LOW_MEMORY_MODE)
print("Maximum body characters used by retrieval:", BODY_MAX_CHARS)

assert artifacts.embeddings.shape == (600_000, 384)
assert artifacts.embeddings.dtype == np.float32
assert artifacts.normalized is True


## 5. Build or restore the BM25 retriever

The BM25 cache is written to Google Drive before the notebook continues. If the runtime disconnects later, the next run restores BM25 instead of rebuilding it. Low-memory mode keeps document bodies but omits duplicated artifact metadata from each in-memory document.


In [ ]:
%%time

import gc
import joblib
import psutil

BM25_CACHE_PATH = (
    DRIVE_ROOT
    / "indices"
    / f"bm25_600k_body{BODY_MAX_CHARS}.joblib"
)
LOCAL_BM25_CACHE_PATH = Path(
    f"/content/bm25_600k_body{BODY_MAX_CHARS}.joblib"
)
BM25_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)


def print_system_memory(label: str) -> None:
    memory = psutil.virtual_memory()
    print(
        f"{label}: used={memory.used / 1024**3:.1f} GB, "
        f"available={memory.available / 1024**3:.1f} GB"
    )


if BM25_CACHE_PATH.exists():
    print("Copying cached BM25 retriever from Drive:", BM25_CACHE_PATH)
    del artifacts
    gc.collect()
    shutil.copy2(BM25_CACHE_PATH, LOCAL_BM25_CACHE_PATH)
    bundle = joblib.load(LOCAL_BM25_CACHE_PATH)
    LOCAL_BM25_CACHE_PATH.unlink(missing_ok=True)

    if bundle.get("document_count") != ARTIFACT_ROW_COUNT:
        raise ValueError("Cached BM25 document count does not match the artifacts")
    if bundle.get("body_max_chars") != BODY_MAX_CHARS:
        raise ValueError("Cached BM25 body limit does not match the configuration")

    bm25 = bundle["retriever"]
    documents = bm25.documents
    del bundle
    gc.collect()
else:
    print("No BM25 cache found; creating aligned search documents...")
    documents = load_search_documents(
        artifacts,
        processed_metadata_path=LOCAL_CLEAN_DATASET,
        max_body_chars=BODY_MAX_CHARS,
        include_metadata=not LOW_MEMORY_MODE,
    )
    assert len(documents) == ARTIFACT_ROW_COUNT

    print("Documents:", len(documents))
    print("Example:", documents[0].model_id, documents[0].title)
    print("Body preview:", documents[0].body[:300])

    del artifacts
    gc.collect()
    print_system_memory("Before BM25 build")

    print("Building BM25 over the complete document collection...")
    bm25 = BM25Retriever(documents)
    print("BM25 build completed")
    print_system_memory("After BM25 build")

    bundle = {
        "format_version": 1,
        "retriever": bm25,
        "document_count": len(documents),
        "body_max_chars": BODY_MAX_CHARS,
        "low_memory_mode": LOW_MEMORY_MODE,
    }

    print("Serializing BM25 cache locally...")
    joblib.dump(
        bundle,
        LOCAL_BM25_CACHE_PATH,
        compress=BM25_CACHE_COMPRESSION,
    )
    cache_size_gb = LOCAL_BM25_CACHE_PATH.stat().st_size / 1024**3
    print(f"Local BM25 cache size: {cache_size_gb:.2f} GB")

    temporary_drive_path = BM25_CACHE_PATH.with_suffix(".joblib.tmp")
    print("Copying BM25 cache to Google Drive...")
    shutil.copy2(LOCAL_BM25_CACHE_PATH, temporary_drive_path)
    temporary_drive_path.replace(BM25_CACHE_PATH)
    LOCAL_BM25_CACHE_PATH.unlink(missing_ok=True)

    print("BM25 cache saved successfully:", BM25_CACHE_PATH)
    del bundle
    gc.collect()

assert len(documents) == ARTIFACT_ROW_COUNT
print("BM25 is ready for", len(documents), "documents")
print_system_memory("BM25 stage completed")


## 6. FAISS index

The vectors are already L2-normalized, so `IndexFlatIP` provides cosine-equivalent ranking. The notebook first tries to load a cached index from Drive. If no index exists, it builds one from the precomputed embeddings and saves it for future runs.


In [ ]:
import gc

# Reload the lightweight artifact handles after the memory-intensive BM25 stage.
artifacts = load_embedding_artifacts(
    LOCAL_EMBEDDING_DIR,
    mmap_embeddings=True,
    expected_dim=ARTIFACT_DIM,
)
artifact_ids = artifacts.ids
dense_metadata = None if LOW_MEMORY_MODE else artifacts.metadata

LOCAL_INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
DRIVE_INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)

source_index = next(
    (path for path in DRIVE_INDEX_CANDIDATES if path.exists()),
    None,
)
if not LOCAL_INDEX_PATH.exists() and source_index is not None:
    print("Copying FAISS index from:", source_index)
    shutil.copy2(source_index, LOCAL_INDEX_PATH)

vector_index = FaissFlatIndex(
    dim=ARTIFACT_DIM,
    metric="inner_product",
)

if LOCAL_INDEX_PATH.exists():
    print("Loading FAISS index:", LOCAL_INDEX_PATH)
    vector_index.load(LOCAL_INDEX_PATH)
else:
    print("Building FAISS index from precomputed embeddings...")
    vector_index.build(artifacts.embeddings)
    vector_index.save(LOCAL_INDEX_PATH)
    shutil.copy2(LOCAL_INDEX_PATH, DRIVE_INDEX_PATH)
    print("Saved index to Drive:", DRIVE_INDEX_PATH)

assert vector_index.ntotal == ARTIFACT_ROW_COUNT
assert vector_index.dim == ARTIFACT_DIM

print("Vectors indexed:", vector_index.ntotal)
print("Metric:", vector_index.metric)
print("Higher is better:", vector_index.higher_is_better)

del artifacts
gc.collect()
print_system_memory("FAISS stage completed")


## 7. Encoder, dense retrieval, and RRF fusion


In [ ]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

from sentence_transformers import SentenceTransformer

encoder_device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = SentenceTransformer(
    ENCODER_MODEL_NAME,
    device=encoder_device,
    cache_folder="/content/huggingface",
)

print("Loaded encoder:", ENCODER_MODEL_NAME)
print("Encoder device:", encoder_device)


In [ ]:
dense = DenseRetriever(
    index=vector_index,
    documents=documents,
    model=encoder,
    encoder_name=ENCODER_MODEL_NAME,
    ids=artifact_ids,
    metadata=dense_metadata,
    normalize_embeddings=NORMALIZE_QUERY_EMBEDDINGS,
)

hybrid = HybridRetriever(
    bm25=bm25,
    dense=dense,
    fusion=ReciprocalRankFusion(k=60),
)


In [ ]:
def results_frame(results):
    return pd.DataFrame(
        [
            {
                "rank": result.rank,
                "model_id": result.model_id,
                "title": result.title,
                "score": result.score,
                "snippet": result.snippet[:240],
            }
            for result in results
        ]
    )


demo_query = Query(
    text="football video analysis model",
    top_k=RETRIEVAL_TOP_K,
)

hybrid_results = hybrid.search(demo_query)
display(results_frame(hybrid_results).head(20))


## 8. Cross-encoder reranking


In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder(
    CROSS_ENCODER_NAME,
    device=encoder_device,
)
reranker = CrossEncoderReranker(
    cross_encoder,
    top_k=RERANK_TOP_K,
)

reranked_results = reranker.rank(demo_query, hybrid_results)
display(results_frame(reranked_results))


In [ ]:
search_pipeline = SearchPipeline(
    retriever=hybrid,
    ranker=reranker,
)


def search_models(text: str, candidate_k: int = RETRIEVAL_TOP_K):
    query = Query(text=text, top_k=candidate_k)
    return results_frame(search_pipeline.search(query))


display(search_models("small multilingual text classification model"))


## 9. Evaluation

Evaluation uses real relevance labels only. The `relevance.csv` file must contain:

- `query`: query text;
- `relevant_model_ids`: either a JSON array of model IDs or IDs separated by `|` or `;`.

If the file is missing, evaluation is skipped without generating artificial labels.


In [ ]:
import ast
import json


def parse_relevant_ids(value) -> tuple[str, ...]:
    if isinstance(value, (list, tuple, set)):
        return tuple(str(item).strip() for item in value if str(item).strip())

    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "null", "[]"}:
        return ()

    if text.startswith("["):
        try:
            return parse_relevant_ids(json.loads(text))
        except json.JSONDecodeError:
            return parse_relevant_ids(ast.literal_eval(text))

    separator = "|" if "|" in text else ";"
    return tuple(part.strip() for part in text.split(separator) if part.strip())


evaluation_examples: list[EvaluationExample] = []

if RELEVANCE_PATH.exists():
    relevance = pd.read_csv(RELEVANCE_PATH)
    required_columns = {"query", "relevant_model_ids"}
    missing = required_columns.difference(relevance.columns)
    if missing:
        raise ValueError(f"Missing relevance columns: {sorted(missing)}")

    evaluation_examples = [
        EvaluationExample(
            query=Query(str(row["query"]), top_k=RETRIEVAL_TOP_K),
            relevant_model_ids=parse_relevant_ids(row["relevant_model_ids"]),
        )
        for _, row in relevance.iterrows()
    ]
    print("Evaluation queries:", len(evaluation_examples))
else:
    print("Evaluation skipped; file not found:", RELEVANCE_PATH)


In [ ]:
if evaluation_examples:
    systems = {
        "bm25": bm25,
        "dense": dense,
        "hybrid_rrf": hybrid,
        "hybrid_rrf_cross_encoder": search_pipeline,
    }

    reports = {}
    for system_name, system in systems.items():
        evaluator = RetrievalEvaluator(
            metric_k_values=(1, 5, 10, 20),
            top_k=RETRIEVAL_TOP_K,
        )
        reports[system_name] = evaluator.evaluate(
            evaluation_examples,
            system,
        )

    metrics_table = pd.DataFrame(
        {
            name: dict(report.metrics)
            for name, report in reports.items()
        }
    ).T

    display(metrics_table)


In [ ]:
if evaluation_examples:
    results_dir = DRIVE_ROOT / "results"
    results_dir.mkdir(parents=True, exist_ok=True)
    report_path = results_dir / "full_search_pipeline_metrics.json"

    with report_path.open("w", encoding="utf-8") as stream:
        json.dump(
            {
                name: {
                    "metrics": dict(report.metrics),
                    "details": dict(report.details),
                }
                for name, report in reports.items()
            },
            stream,
            ensure_ascii=False,
            indent=2,
        )

    print("Saved evaluation report:", report_path)


## Ready for demonstration

A concise presentation flow is:

1. show artifact validation and the `(600000, 384)` matrix;
2. load the cached FAISS index;
3. show hybrid RRF results before reranking;
4. show the ranking change after the cross-encoder;
5. present the Recall/MRR/nDCG and latency comparison using real labels.
